In [2]:
import polars as pl
import plotly.express as px
import os, sys

module_path = os.path.abspath(os.path.join('../../../..')) # or the path to your source code
sys.path.insert(0, module_path)
import util

config = util.summary_settings
input_config = util.input_settings

In [3]:
person = util.get_person_data()
tour = util.get_tour_data()

## number of tours per person

In [4]:
df_tour = tour.join(person, on=['hhno','pno','source'], how='left')

In [5]:
df_plot = df_tour.group_by('source').agg([
    pl.sum('toexpfac').alias('toexpfac_sum')
]).join(
    person.group_by('source').agg([
        pl.sum('psexpfac').alias('psexpfac_sum')
    ]),
    on='source',
    how='left'
).with_columns(
    (pl.col('toexpfac_sum') / pl.col('psexpfac_sum')).alias('average tours'),
    pl.lit('').alias('person')
)

fig = px.bar(df_plot, x="person", y="average tours", color="source",
             barmode="group", title="number of tours per person")
fig.update_layout(height=400, width=400, font=dict(size=11),
                  xaxis=dict(dtick=1, categoryorder='category ascending'),
                  yaxis=dict(tickformat=".3"))
fig.show()

## percent of tours by purpose

In [14]:
df_plot = df_tour.group_by(['source', 'pdpurp_label']).agg([
    pl.sum('toexpfac').alias('toexpfac_sum')
]).with_columns(
    (pl.col('toexpfac_sum') / pl.col('toexpfac_sum').sum().over('source')).alias('percentage')
)

df_plot_ct = df_tour.group_by(['source', 'pdpurp_label']).agg([
    pl.count('toexpfac').alias('sample count')
])

df_plot = df_plot.join(df_plot_ct, on=['source', 'pdpurp_label'], how='left').sort(['source', 'pdpurp_label'])

fig = px.bar(df_plot, x="pdpurp_label", y="percentage", color="source",
             barmode="group", hover_data=['sample count'], title="tour purpose")
fig.update_layout(height=400, width=700, font=dict(size=11),
                  yaxis=dict(tickformat=".2%"))
fig.show()

## number of tours per person by segment

In [16]:
def plot_segment(df, tour_group_var, person_group_var, title_name):
    fig = px.bar(df, x=tour_group_var[1], y="average tour", color="source",
                 barmode="group", title=title_name)
    fig.update_layout(height=400, width=700, font=dict(size=11),
                      yaxis=dict(tickformat=".2"))
    fig.show()

df_plot = df_tour.group_by(['source', 'pdpurp_label']).agg([
    pl.sum('toexpfac').alias('toexpfac_sum')
]).join(
    person.group_by(['source']).agg([
        pl.sum('psexpfac').alias('psexpfac_sum')
    ]),
    on='source',
    how='left'
).with_columns(
    (pl.col('toexpfac_sum') / pl.col('psexpfac_sum')).alias('average tour')
).sort(['source', 'pdpurp_label'])

plot_segment(df_plot, tour_group_var=['source', 'pdpurp_label'], person_group_var=['source'],
             title_name="number of tours per person by tour purpose")


In [17]:
# plot_segment(tour_group_var=['source','pptyp_label'],person_group_var=['source','pptyp_label'],
#              title_name="number of tours per person by person type")

df_plot = df_tour.group_by(['source', 'pptyp_label']).agg([
    pl.sum('toexpfac').alias('toexpfac_sum')
]).join(
    person.group_by(['source', 'pptyp_label']).agg([
        pl.sum('psexpfac').alias('psexpfac_sum')
    ]),
    on=['source', 'pptyp_label'],
    how='left'
).with_columns(
    (pl.col('toexpfac_sum') / pl.col('psexpfac_sum')).alias('average tour')
).sort(['source', 'pptyp_label'])

plot_segment(df_plot, tour_group_var=['source', 'pptyp_label'], person_group_var=['source', 'pptyp_label'],
             title_name="number of tours per person by person type")

In [18]:
wk_tour = df_tour.filter(pl.col('pdpurp') == 1)

df_plot = wk_tour.group_by(['source', 'pptyp_label']).agg([
    pl.sum('toexpfac').alias('toexpfac_sum')
]).join(
    person.group_by(['source', 'pptyp_label']).agg([
        pl.sum('psexpfac').alias('psexpfac_sum')
    ]),
    on=['source', 'pptyp_label'],
    how='left'
).with_columns(
    (pl.col('toexpfac_sum') / pl.col('psexpfac_sum')).alias('average tour')
).sort(['source', 'pptyp_label'])

fig = px.bar(df_plot, x='pptyp_label', y="average tour", color="source",
             barmode="group", title="number of work tours per person by person type")
fig.update_layout(height=400, width=700, font=dict(size=11),
                  yaxis=dict(tickformat=".2"))
fig.show()

### Tour by Purpose and Person Type

In [19]:

def plot_by_pptyp(df_tour, person_type):
    df_plot = df_tour.filter(
        pl.col('pptyp') == int(person_type)
    ).group_by(['source', 'pdpurp_label']).agg([
        pl.sum('toexpfac').alias('toexpfac_sum')
    ]).join(
        person.filter(pl.col('pptyp') == int(person_type)).group_by(['source']).agg([
            pl.sum('psexpfac').alias('psexpfac_sum')
        ]),
        on=['source'],
        how='left'
    ).with_columns(
        (pl.col('toexpfac_sum') / pl.col('psexpfac_sum')).alias('average tour')
    ).sort(['source', 'pdpurp_label'])

    plot_segment(df_plot, tour_group_var=['source', 'pdpurp_label'], person_group_var=['source'],
                 title_name="number of tours per person for person type " + str(person_type))


In [20]:
plot_by_pptyp(df_tour, '1')

In [21]:
plot_by_pptyp(df_tour, '2')

In [22]:
plot_by_pptyp(df_tour, '3')

In [23]:
plot_by_pptyp(df_tour, '4')

In [24]:
plot_by_pptyp(df_tour, '5')

In [25]:
plot_by_pptyp(df_tour, '6')

In [26]:
plot_by_pptyp(df_tour, '7')

In [27]:
plot_by_pptyp(df_tour, '8')